In [1]:
from metrics.distance_metrics import distance_to_closest_record, nearest_neighbor_distance_ratio, cosine_similarity_metric,outliers_similarity, exact_matches, hausdorff_distance
import pandas as pd

In [2]:
real_data=pd.read_csv('../notebooks/data/adult_complete.csv')
synthetic_data=pd.read_csv('../notebooks/data/synthetic_data.csv')

In [3]:
categorical_columns = ['workclass','education-num','marital-status','occupation','relationship','race','sex','native-country','income']

In [7]:
exact_matches(real_data, synthetic_data, 0.1)

0.9999692884125181

In [5]:
hausdorff_distance(real_data,synthetic_data,categorical_columns)

2.65

In [6]:
distance_to_closest_record(real_data,synthetic_data,categorical_columns)

DCRResult(min_dcr=0.002, mean_dcr=0.429, std_dcr=0.578)

In [7]:
nearest_neighbor_distance_ratio(real_data,synthetic_data,categorical_columns)

NNDRResult(min_nndr=0.013, mean_nndr=0.785, std_nndr=0.256)

In [8]:
cosine_similarity_metric(real_data,synthetic_data,categorical_columns)

CSResult(mean_cs=0.447, std_cs=0.176, max_cs=1.0)

In [4]:
from attacks.attribute_disclosure import attribute_disclosure_as_ml_task

attribute_disclosure_as_ml_task(real_data,synthetic_data,['education-num','workclass','occupation'],key_length=3,target='income',categorical_columns=categorical_columns)

{'acc':                Dummy  RF_mean  RF_std  SVM_mean  SVM_std  NB_mean  NB_std   
 Original_acc   0.759    0.795     0.0     0.786      0.0    0.773     0.0  \
 Synthetic_acc  0.759    0.757     0.0     0.759      0.0    0.759     0.0   
 
                KNN_mean  KNN_std  LR_mean  LR_std  ENS_mean  ENS_std   
 Original_acc      0.735      0.0    0.779     0.0     0.784      0.0  \
 Synthetic_acc     0.704      0.0    0.759     0.0     0.759      0.0   
 
                ROW_mean  ROW_std  
 Original_acc      0.775      0.0  
 Synthetic_acc     0.750      0.0  ,
 'f1':               Dummy  RF_mean  RF_std  SVM_mean  SVM_std  NB_mean  NB_std   
 Original_f1     0.0    0.428     0.0     0.339      0.0    0.245     0.0  \
 Synthetic_f1    0.0    0.016     0.0     0.000      0.0    0.000     0.0   
 
               KNN_mean  KNN_std  LR_mean  LR_std  ENS_mean  ENS_std  ROW_mean   
 Original_f1      0.367      0.0    0.304     0.0       0.3      0.0     0.330  \
 Synthetic_f1     0.169 

In [ ]:
outliers_similarity(real_data,synthetic_data,categorical_columns,method='dbscan')

In [5]:
from sklearn.neighbors import BallTree
from sklearn import metrics
from collections import Counter
from utils.aux_functions import most_frequent

In [6]:
def GCAPClassifier(X_train, y_train, X_test, key):
    '''Algorithm proposed in https://doi.org/10.1145/3374664.3375722 '''

    bat = BallTree(X_train, metric='hamming')
    y_pred = []
    for entry in range(len(X_test)):
        pt = X_test.values[entry].reshape(1, -1)
        for i in range(len(key) + 1):
            match_list = bat.query_radius(pt, r=i / len(key))[0]
            if len(match_list) != 0:
                break
            
        maj_list = []
        for i in range(len(match_list)):
            maj_list.append(y_train[match_list[i]])
            
            
        y_pred.append(most_frequent(maj_list))

    return y_pred

In [7]:
key_length = 3
quasi_identifiers = ['marital-status','income','education-num']
target = 'sex'

In [11]:
import itertools
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from utils.aux_functions import majority_voting


key_list = list(itertools.combinations(quasi_identifiers, key_length))
    
# The original dataset is used as reference (Upper Bound)
data_evaluated = {
    'Original': real_data,
    'Synthetic':synthetic_data
}

# Define classifiers used in the task

estimators = {
    'RF': RandomForestClassifier(),
    'SVM': SVC(),
    'NB':  GaussianNB(),
    'KNN': KNeighborsClassifier(),
    'LR':  LogisticRegression()
}

# The dummy classifier is used as reference (Lower Bound)
baseline =  DummyClassifier()

# Preprocessing step in case is not defined. 
numerical_columns= [ col for col in real_data.columns if col not in categorical_columns]

# Preprocess numerical columns
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Preprocess categorical columns
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Build transformer with preprocessing pipelines.
preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", num_transformer, numerical_columns),
        ("categorical", cat_transformer, categorical_columns),
    ]
)   

results_acc = pd.DataFrame(index=data_evaluated.keys())
results_f1 = pd.DataFrame(index=data_evaluated.keys())
        
for data_name,dataset in data_evaluated.items():

    train_data=pd.DataFrame(preprocessor.fit_transform(dataset),columns=numerical_columns+categorical_columns)
    test_data=pd.DataFrame(preprocessor.transform(real_data),columns=numerical_columns+categorical_columns)
    
    # Keys combinations    
    for i,key in enumerate(key_list):

        X_train = train_data[list(key)]
        X_test = test_data[list(key)]
        
        y_train = train_data.loc[:,target].values          
        y_test = test_data.loc[:,target].values
        
        y_pred=GCAPClassifier(X_train,y_train,X_test,key)
        print(round(metrics.accuracy_score(y_test, y_pred), 3))


0.732
0.723


In [ ]:
from sklearn.neighbors import RadiusNeighborsClassifier
neigh = RadiusNeighborsClassifier(radius=1,algorithm='ball_tree',metric='hamming')
neigh.fit(X_train, y_train)
y_pred=neigh.predict(X_test)
round(metrics.accuracy_score(y_test, y_pred), 3)

In [ ]:
synthetic_data